In [1]:
# Get daily constraint ranked by the abs RT - DA 
from functions import (
    get_recent_constraint_mvalue,
    get_node_dfax_from_constraint_num,
    get_hourly_mvalue_for_constraint_num,
    get_constraint_node_prices_from_constraint_num,
    get_constraints_node_price,
    get_metrics_for_nodes,
    get_recent_direction_accuracy,
    get_weather_date,
    predict_tomorrow_percentile,
    analyze_constraint_by_zone,
    plot_constraint_seasonality,
    plot_recent_hourly_distribution,
)
import pandas as pd

In [2]:
now = pd.Timestamp.now(tz='US/Central')
days_ahead = 2 if now.hour >= 10 else 1
bid_dt = (now + pd.Timedelta(days=days_ahead)).strftime('%Y-%m-%d')
print(bid_dt)

2026-06-24


In [3]:
dt = (pd.Timestamp(bid_dt) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
mvalue= get_recent_constraint_mvalue(dt, dt, threshold=500)
dfax= get_node_dfax_from_constraint_num(mvalue)
hourly_mvalue = get_hourly_mvalue_for_constraint_num(mvalue)
fundamentals = get_weather_date(hourly_mvalue, dt)

start_dt: 2026-06-23, end_dt: 2026-06-23, Market: SPP
Fetching RT mvalues...
Fetching DA mvalues...
RT constraints: 6, DA constraints: 41
Fetching constraint details...


,oops_constraint_num,rt_total,da_total,rt_da,abs_rt_da_diff,monitored,contingency
0,649753,0.0000,-1028.1254,1028.1254,1028.1254,lnrussett-sbrown,okge:brown2bodlecaney1:138:7:34
1,776592,0.0000,-849.4255,849.4255,849.4255,xfmrduncan-duncan,wfec:anadarkofletch3comanch2:138
2,775436,0.0000,-726.0008,726.0008,726.0008,xfmrsiouxcy-siouxcy,waue:siouxcyxfkv5a3:23016113.8
11,833578,0.0000,-122.9510,122.9510,122.9510,lnrussett-sbrown,okgewfec:bison_okhugopp4:345:1:1
37,653411,-11.7358,0.0000,-11.7358,11.7358,lnrussett-sbrown,okge:brown2bodlecaney1:138:7:34
42,832846,-7.3353,0.0000,-7.3353,7.3353,lnrussett-sbrown,okgewfec:bison_okhugopp4:345:1:1


RT active constraint-days: 373
DA active constraint-days: 712
   oops_constraint_num          dt  hr  rt_mvalue  da_mvalue     rt_da
0               649753  2024-07-03   2        0.0   -39.5694   39.5694
1               649753  2024-07-03   3        0.0   -32.8592   32.8592
2               649753  2024-07-03   4        0.0   -30.0269   30.0269
3               649753  2024-07-11  22        0.0  -250.5099  250.5099
4               649753  2024-07-11  23        0.0  -303.0936  303.0936

Merged hourly rows: 9880


,oops_constraint_num,dt,hr,rt_mvalue,da_mvalue,rt_da,monitored,contingency
0,649753,2024-07-03,2,0.0,-39.5694,39.5694,lnrussett-sbrown,okge:brown2bodlecaney1:138:7:34
1,649753,2024-07-03,3,0.0,-32.8592,32.8592,lnrussett-sbrown,okge:brown2bodlecaney1:138:7:34
2,649753,2024-07-03,4,0.0,-30.0269,30.0269,lnrussett-sbrown,okge:brown2bodlecaney1:138:7:34
3,649753,2024-07-11,22,0.0,-250.5099,250.5099,lnrussett-sbrown,okge:brown2bodlecaney1:138:7:34
4,649753,2024-07-11,23,0.0,-303.0936,303.0936,lnrussett-sbrown,okge:brown2bodlecaney1:138:7:34
...,...,...,...,...,...,...,...,...
9875,833578,2026-06-23,14,0.0,-16.5518,16.5518,lnrussett-sbrown,okgewfec:bison_okhugopp4:345:1:1
9876,833578,2026-06-23,15,0.0,-24.7455,24.7455,lnrussett-sbrown,okgewfec:bison_okhugopp4:345:1:1
9877,833578,2026-06-23,16,0.0,-1.9657,1.9657,lnrussett-sbrown,okgewfec:bison_okhugopp4:345:1:1
9878,833578,2026-06-23,17,0.0,-19.7561,19.7561,lnrussett-sbrown,okgewfec:bison_okhugopp4:345:1:1


Reserve zones available: [1, 2, 3, 4, 5, 21]
Wind cols: ['rz1_spp_res_zonal_wind_forecast_f', 'rz2_spp_res_zonal_wind_forecast_f', 'rz3_spp_res_zonal_wind_forecast_f', 'rz4_spp_res_zonal_wind_forecast_f', 'rz5_spp_res_zonal_wind_forecast_f', 'rz21_spp_res_zonal_wind_forecast_f']
Load cols: ['rz1_spp_res_zonal_load_forecast_f', 'rz2_spp_res_zonal_load_forecast_f', 'rz3_spp_res_zonal_load_forecast_f', 'rz4_spp_res_zonal_load_forecast_f', 'rz5_spp_res_zonal_load_forecast_f', 'rz21_spp_res_zonal_load_forecast_f']


In [4]:
# from IPython.display import display, HTML
# row_counts = hourly_mvalue.groupby('monitored').size().sort_values()
# print(row_counts.to_string())

# for name in row_counts.index:
#     display(HTML(f'''
#     <hr style="border: 3px solid black; margin: 30px 0;">
#     <h1 style="background:#2c3e50; color:white; padding:12px 20px; border-radius:6px; font-family:monospace;">
#         {name} &nbsp;<span style="font-size:0.6em; color:#aaa;">({row_counts[name]:,} rows)</span>
#     </h1>
#     <hr style="border: 1px solid #aaa; margin-bottom: 20px;">
#     '''))

#     analyze_constraint_by_zone(fundamentals, hourly_mvalue, name, 4, 95)
#     plot_constraint_seasonality(hourly_mvalue, name)
#     plot_recent_hourly_distribution(hourly_mvalue, name, bid_dt, days=10)
#     table = get_constraints_node_price(hourly_mvalue, dfax, [name])
#     get_metrics_for_nodes(table)

In [5]:
# name = 'lnraun-tekamho'
# zone= 1
# analyze_constraint_by_zone(fundamentals, hourly_mvalue, name, zone, 90)
# plot_constraint_seasonality(hourly_mvalue, name)
# table = get_constraints_node_price(hourly_mvalue, dfax, [name])
# get_metrics_for_nodes(table)

In [6]:
fundamentals = get_weather_date(hourly_mvalue, bid_dt)

Reserve zones available: [1, 2, 3, 4, 5, 21]
Wind cols: ['rz1_spp_res_zonal_wind_forecast_f', 'rz2_spp_res_zonal_wind_forecast_f', 'rz3_spp_res_zonal_wind_forecast_f', 'rz4_spp_res_zonal_wind_forecast_f', 'rz5_spp_res_zonal_wind_forecast_f', 'rz21_spp_res_zonal_wind_forecast_f']
Load cols: ['rz1_spp_res_zonal_load_forecast_f', 'rz2_spp_res_zonal_load_forecast_f', 'rz3_spp_res_zonal_load_forecast_f', 'rz4_spp_res_zonal_load_forecast_f', 'rz5_spp_res_zonal_load_forecast_f', 'rz21_spp_res_zonal_load_forecast_f']


In [7]:
tomorrow_pct = predict_tomorrow_percentile(fundamentals, bid_dt)


--- Zone 1 ---


,hr,wind_value,wind_pct,load_value,load_pct
0,1,362.7,3.2,4117.0,48.4
1,2,385.1,3.2,3990.7,45.2
2,3,408.1,6.5,3842.6,45.2
3,4,419.2,9.7,3867.6,51.6
4,5,421.6,6.5,3930.0,58.1
5,6,452.3,9.7,4102.2,74.2
6,7,523.5,19.4,4388.1,80.6
7,8,524.3,29.0,4653.1,80.6
8,9,392.2,19.4,4812.7,80.6
9,10,293.7,9.7,4966.7,80.6



--- Zone 2 ---


,hr,wind_value,wind_pct,load_value,load_pct
0,1,2601.3,16.1,718.6,74.2
1,2,2405.7,19.4,681.1,71.0
2,3,2204.6,19.4,665.8,71.0
3,4,2220.1,22.6,665.0,74.2
4,5,2247.5,25.8,670.4,77.4
5,6,2163.5,29.0,715.9,93.5
6,7,2016.3,29.0,752.9,93.5
7,8,1775.2,22.6,771.5,96.8
8,9,1569.2,22.6,792.5,87.1
9,10,1452.1,22.6,805.5,83.9



--- Zone 3 ---


...


--- Zone 5 ---


,hr,wind_value,wind_pct,load_value,load_pct
0,1,1527.7,25.8,3645.7,61.3
1,2,1483.1,22.6,3609.0,74.2
2,3,1417.3,22.6,3534.6,45.2
3,4,1394.3,16.1,3524.9,48.4
4,5,1402.6,16.1,3521.8,45.2
5,6,1375.3,19.4,3543.2,41.9
6,7,1286.2,22.6,3661.9,48.4
7,8,1160.2,22.6,3787.2,41.9
8,9,1023.4,25.8,3925.2,54.8
9,10,965.4,22.6,3960.0,38.7



--- Zone 21 ---


,hr,wind_value,wind_pct,load_value,load_pct
0,1,331.2,9.7,3178.1,74.2
1,2,354.1,12.9,2993.5,74.2
2,3,283.1,9.7,2861.1,74.2
3,4,247.3,12.9,2774.2,77.4
4,5,234.1,12.9,2709.3,77.4
5,6,233.9,19.4,2679.1,77.4
6,7,218.1,19.4,2656.9,61.3
7,8,171.2,19.4,2665.4,48.4
8,9,161.9,19.4,2731.8,48.4
9,10,185.2,19.4,2840.3,61.3



--- Summary (median pct by zone / peak) ---
1: ow: 12, fw: 8, ol: 71, fl: 54
2: ow: 9, fw: 21, ol: 80, fl: 75
3: ow: 33, fw: 91, ol: 96, fl: 93
4: ow: 9, fw: 25, ol: 75, fl: 67
5: ow: 24, fw: 21, ol: 56, fl: 46
21: ow: 35, fw: 12, ol: 69, fl: 74
